In [2]:
# AgentOps_Practical_Implementation.ipynb
#
# This Jupyter notebook demonstrates a multi-agent system for stock analysis
# using CrewAI, integrated with AgentOps for enhanced observability and monitoring.
# The system utilizes specialized agents to gather financial news, analyze stock
# price trends, and generate investor-friendly reports.
#
# This implementation showcases how to build intelligent, observable agent systems for
# complex tasks like financial analysis.

import os
from getpass import getpass

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM
from crewai.tools import BaseTool
import yfinance as yf
from duckduckgo_search import DDGS
import agentops
import sys

# Set the Python I/O encoding to UTF-8 to handle various character sets.
os.environ["PYTHONIOENCODING"] = "utf-8"

# Load environment variables from the local .env file if present.
load_dotenv()

# Configure the Gemini API key. Prompt the user if it is not set in the environment.
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    api_key = getpass("Enter your Gemini API key: ")
os.environ["GEMINI_API_KEY"] = api_key

# Load AgentOps API key if not already set.
agentops_api_key = os.getenv("AGENTOPS_API_KEY")
if not agentops_api_key:
    agentops_api_key = getpass("Enter your AgentOps API key: ")
os.environ["AGENTOPS_API_KEY"] = agentops_api_key

# Initialize AgentOps with the retrieved API key for monitoring and observability.
agentops.init(api_key=os.environ["AGENTOPS_API_KEY"])

# Initialize the CrewAI LLM client for Gemini.
# Using 'gemini/gemini-2.5-flash' model with a temperature of 0.7 for balanced creativity and factual adherence.
llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=api_key,
    temperature=0.7,
)


class StockSearchTool(BaseTool):
    """
    A tool to search for the latest news and updates about a stock using DuckDuckGo.
    Inherits from CrewAI's BaseTool to integrate with agent workflows.
    """
    name: str = "StockNewsSearcher"
    description: str = "Search for the latest news and updates about a stock using DuckDuckGo"

    def _run(self, query: str) -> str:
        """
        Executes the stock news search.
        Args:
            query (str): The search query, typically a stock ticker or company name.
        Returns:
            str: A string containing summarized news highlights or an error message.
        """
        # Use DuckDuckGo to gather a compact set of recent news results for the requested stock.
        try:
            with DDGS() as ddgs:
                # Retrieve a maximum of 2 results to keep the output concise.
                results = ddgs.text(query, max_results=2)
            if not results:
                return "No news results were found."
            # Join the 'body' of the search results into a single string.
            return "\n".join([r.get("body", "") for r in results if r.get("body")])
        except Exception as exc:
            return f"News search failed: {exc}"


class YahooFinanceTool(BaseTool):
    """
    A tool to fetch the latest 1-month stock price history for a given ticker using yFinance.
    Inherits from CrewAI's BaseTool for seamless integration into agent tasks.
    """
    name: str = "YahooFinanceFetcher"
    description: str = "Get the latest 1-month stock price history for a given ticker using yFinance"

    def _run(self, ticker: str) -> str:
        """
        Fetches stock price history for a given ticker.
        Args:
            ticker (str): The stock ticker symbol (e.g., 'AAPL').
        Returns:
            str: A string representation of the last 3 days of stock price history or an error message.
        """
        # Fetch the most recent price history to support the price analysis task.
        try:
            stock = yf.Ticker(ticker)
            # Retrieve 1 month of historical data.
            hist = stock.history(period="1mo")
            if hist.empty:
                return f"No price data available for {ticker}."
            # Return the last 3 entries of the historical data for a concise overview.
            return hist.tail(3).to_string()
        except Exception as exc:
            return f"Price data lookup failed: {exc}"


# Define the two specialized agents for the multi-agent workflow.

# Stock Analyst Agent: Specializes in financial data analysis.
stock_analyst = Agent(
    role="Stock Analyst",
    goal="Analyze recent stock data and news",
    backstory="Expert in financial trends, macro indicators, and company performance",
    verbose=True,  # Enable verbose output for detailed agent execution logs.
    allow_delegation=False,  # This agent does not delegate its tasks.
    llm=llm,  # Assign the initialized Gemini LLM.
    agentops_enabled=True,  # Enable AgentOps for this agent.
)

# Report Writer Agent: Focuses on generating clear, investor-friendly reports.
report_writer = Agent(
    role="Report Generator",
    goal="Write investor-friendly summaries of stock analysis",
    backstory="Professional writer with expertise in finance reporting",
    verbose=True,  # Enable verbose output for detailed agent execution logs.
    allow_delegation=False,  # This agent does not delegate its tasks.
    llm=llm,  # Assign the initialized Gemini LLM.
    agentops_enabled=True,  # Enable AgentOps for this agent.
)

# Create tasks for news gathering, trend analysis, and report generation.

# Initialize instances of the custom tools.
search_tool = StockSearchTool()
finance_tool = YahooFinanceTool()

# Task 1: Search for latest news about AAPL stock.
search_task = Task(
    description="Search latest news and updates about the stock 'AAPL' using DuckDuckGo.",
    expected_output="Summarized news highlights for Apple stock.",
    agent=stock_analyst,  # Assigned to the stock_analyst agent.
    tools=[search_tool],  # Utilizes the StockSearchTool.
)

# Task 2: Analyze Apple stock price trends.
analysis_task = Task(
    description="Analyze Apple stock price trends using yFinance.",
    expected_output="Key trends and technical highlights for the past month.",
    agent=stock_analyst,  # Also assigned to the stock_analyst agent.
    tools=[finance_tool],  # Utilizes the YahooFinanceTool.
)

# Task 3: Write an investor report based on the analysis and news.
report_task = Task(
    description="Write a clean investor report using previous analysis and news insights.",
    expected_output="Concise report with market summary and investment outlook.",
    agent=report_writer,
    # This task doesn't directly use tools; it processes output from previous tasks.
)

# Assemble the crew so the agents can work together in sequence.
crew = Crew(
    agents=[stock_analyst, report_writer],  # Define the agents participating in the crew.
    tasks=[search_task, analysis_task, report_task],  # Define the sequence of tasks.
    verbose=False,
)

In [3]:
# Run the agent crew and print the final investor report.
# The kickoff_async method is used to run the crew asynchronously.
result = await crew.kickoff_async()

print("\n📊 Final Stock Analysis Report:\n")
print(result)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst                                                                                           │
│                                                                                                                 │
│  Task: Search latest news and updates about the stock 'AAPL' using DuckDuckGo.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

C:\Users\Shubham\AppData\Local\Temp\ipykernel_6388\180449302.py:70: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Tool stock_news_searcher executed with result: Find the latest Apple Inc. (AAPL) stock quote, history, news and other vital information to help you with your stock trading and investing.
Apple Inc. is an American multinational technology company h...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Find the latest Apple Inc. (AAPL) stock quote, history, news and other vital information to help you with      │
│  your stock trading and investing.                                                                              │
│  Apple Inc. is an American multinational technology company headquartered in Cupertino, California, in Silicon  │
│  Valley, and known for …                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1619: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1619: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1626: DeprecationWarning: deprecated
  and hasattr(agent, "multimodal")
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1627: DeprecationWarning: deprecated
  and getattr(agent, "multimodal", False)


...

c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1619: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1619: DeprecationWarning: deprecated
  if hasattr(agent, "allow_code_execution") and getattr(
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1626: DeprecationWarning: deprecated
  and hasattr(agent, "multimodal")
c:\Shubham\ProjectsAgenticAI\StockAnalysisCrewAI_AgentOps\sa_agentops_venv\Lib\site-packages\crewai\crew.py:1627: DeprecationWarning: deprecated
  and getattr(agent, "multimodal", False)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Generator                                                                                        │
│                                                                                                                 │
│  Task: Write a clean investor report using previous analysis and news insights.                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Generator                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Apple Inc. (AAPL) Investor Report**                                                                          │
│                                                                                                                 │
│  **Reporting Period:** July 15, 2026 – July 17, 2026                                                            │
│                                                                                                                 │
│  **Market Summary**                                                                                             │
│  Apple Inc. (AAPL), the multinational technology giant, demonstrated a strong bullish trend during the period   │
│  of July 15th to July 17th, 2026. The stock consistently posted upward price movements, supported by            │
│  increasing trading volumes, signaling robust investor confidence and buying pressure. Over these three days,   │
│  AAPL's closing price advanced from $327.50 to $333.74, a gain of approximately 1.9%. This price appreciation   │
│  was accompanied by a noticeable increase in daily trading volume, climbing from 60.96 million shares to 63.37  │
│  million shares, reinforcing the validity of the upward movement.                                               │
│                                                                                                                 │
│  **Key Performance Highlights**                                                                                 │
│                                                                                                                 │
│  *   **Consistent Price Appreciation:** AAPL's closing price showed a steady increase each day, moving from     │
│  $327.50 on July 15th to $333.26 on July 16th, and finally to $333.74 on July 17th. This sustained upward       │
│  trajectory suggests a strong positive sentiment surrounding the stock.                                         │
│  *   **Increasing Trading Volume:** The daily trading volume followed suit, rising from 60,957,600 on July      │
│  15th to 62,970,600 on July 16th, and peaking at 63,365,300 on July 17th. This trend, where rising prices are   │
│  backed by increasing volume, is a classic bullish indicator, suggesting broad market participation in the      │
│  stock's ascent.                                                                                                │
│  *   **Intraday Strength:** Despite typical intraday volatility, the stock consistently managed to close        │
│  strong, often significantly higher than its opening price or maintaining gains. On July 15th, AAPL opened      │
│  lower but closed significantly higher, indicating strong end-of-day buying. On July 17th, it opened lower      │
│  than the previous close but still achieved the highest price point of the period ($334.99) before closing      │
│  higher than its open.                                                                                          │
│                                                                                                                 │
│  **Investment Outlook**                                                                                         │
│  The recent performance of Apple Inc. (AAPL) from July 15th to July 17th, 2026, presents a clear short-term     │
│  bullish outlook. The concurrent rise in both stock pri


📊 Final Stock Analysis Report:

**Apple Inc. (AAPL) Investor Report**

**Reporting Period:** July 15, 2026 – July 17, 2026

**Market Summary**
Apple Inc. (AAPL), the multinational technology giant, demonstrated a strong bullish trend during the period of July 15th to July 17th, 2026. The stock consistently posted upward price movements, supported by increasing trading volumes, signaling robust investor confidence and buying pressure. Over these three days, AAPL's closing price advanced from $327.50 to $333.74, a gain of approximately 1.9%. This price appreciation was accompanied by a noticeable increase in daily trading volume, climbing from 60.96 million shares to 63.37 million shares, reinforcing the validity of the upward movement.

**Key Performance Highlights**

*   **Consistent Price Appreciation:** AAPL's closing price showed a steady increase each day, moving from $327.50 on July 15th to $333.26 on July 16th, and finally to $333.74 on July 17th. This sustained upward trajector